# Virtual try-on — ER · FLUX.2 klein 4B

## 1 · Install

In [ ]:
import glob
import os
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError("no GPU: Runtime -> Change runtime type -> A100")
if torch.cuda.get_device_properties(0).total_memory < 20 * 1024 ** 3:
    raise RuntimeError(f"{torch.cuda.get_device_name(0)} has too little memory; use an A100 (40 GB)")

PIP = [sys.executable, "-m", "pip"]
subprocess.run(PIP + ["install", "-q", "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
                      "sentencepiece", "protobuf", "mediapipe==1.0.1"], check=True, capture_output=True)
subprocess.run(PIP + ["uninstall", "-q", "-y", "onnxruntime", "onnxruntime-gpu"], capture_output=True)
ORT_CUDA_PROBE = """
import ctypes, glob, os, site, onnxruntime
for d in sorted({d for p in site.getsitepackages() for d in glob.glob(os.path.join(p, "nvidia", "*", "lib"))}):
    for f in os.listdir(d):
        if ".so" in f and any(k in f for k in ("cudart", "cublas", "cudnn", "cufft", "curand")):
            try:
                ctypes.CDLL(os.path.join(d, f), mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass
ctypes.CDLL(glob.glob(os.path.dirname(onnxruntime.__file__) + "/capi/libonnxruntime_providers_cuda.so")[0],
            mode=ctypes.RTLD_GLOBAL)
"""
for spec in ["", "==1.22.0", "==1.21.1", "==1.20.1", "==1.19.2", "==1.18.1"]:
    if (subprocess.run(PIP + ["install", "-q", f"onnxruntime-gpu{spec}"], capture_output=True).returncode == 0
            and subprocess.run([sys.executable, "-c", ORT_CUDA_PROBE], capture_output=True).returncode == 0):
        break
else:
    raise RuntimeError("no onnxruntime-gpu build loads CUDA on this runtime")
subprocess.run(PIP + ["uninstall", "-q", "-y", "opencv-python", "opencv-python-headless", "opencv-contrib-python"],
               capture_output=True)
subprocess.run(PIP + ["install", "-q", "opencv-contrib-python-headless==5.0.0.93"], check=True, capture_output=True)

## 2 · Downloads

In [ ]:
import hashlib
import urllib.request

from huggingface_hub import hf_hub_download

MODEL_ROOT = "/content/models"
BFL = ("black-forest-labs/FLUX.2-klein-4B", "e7b7dc27f91deacad38e78976d1f2b499d76a294", "FLUX.2-klein-4B")
PHOTOROOM = ("Photoroom/FLUX.2-klein-4b-fp8-diffusers", "408c457f3589e17a1be1dae5bf0dcaf09cd4985f",
             "FLUX.2-klein-4b-fp8-diffusers")
KLEIN_FILES = {
    BFL: ["model_index.json", "scheduler/scheduler_config.json",
          "text_encoder/config.json", "text_encoder/generation_config.json",
          "text_encoder/model.safetensors.index.json",
          "text_encoder/model-00001-of-00002.safetensors", "text_encoder/model-00002-of-00002.safetensors",
          "tokenizer/added_tokens.json", "tokenizer/chat_template.jinja", "tokenizer/merges.txt",
          "tokenizer/special_tokens_map.json", "tokenizer/tokenizer.json", "tokenizer/tokenizer_config.json",
          "tokenizer/vocab.json", "vae/config.json", "vae/diffusion_pytorch_model.safetensors"],
    PHOTOROOM: ["transformer_bf16/config.json", "transformer_bf16/diffusion_pytorch_model.safetensors"],
}
CROP_FILES = {
    "BiRefNet_lite.onnx": (
        "https://huggingface.co/onnx-community/BiRefNet_lite-ONNX/resolve/"
        "de15b22ba131738a16dff04aab8bdf8dc32e3ac1/onnx/model.onnx",
        "5600024376f572a557870a5eb0afb1e5961636bef4e1e22132025467d0f03333"),
    "parsing_atr.onnx": (
        "https://huggingface.co/basso4/humanparsing/resolve/"
        "4fd18f98561bae00b5c24342c92307b4780b2a8d/parsing_atr.onnx",
        "04c7d1d070d0e0ae943d86b18cb5aaaea9e278d97462e9cfb270cbbe4cd977f4"),
    "selfie_multiclass_256x256.tflite": (
        "https://storage.googleapis.com/mediapipe-models/image_segmenter/"
        "selfie_multiclass_256x256/float32/latest/selfie_multiclass_256x256.tflite",
        "c6748b1253a99067ef71f7e26ca71096cd449baefa8f101900ea23016507e0e0"),
    "pose_landmarker_lite.task": (
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
        "pose_landmarker_lite/float16/1/pose_landmarker_lite.task",
        "59929e1d1ee95287735ddd833b19cf4ac46d29bc7afddbbf6753c459690d574a"),
}


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()


for (repo, revision, folder), names in KLEIN_FILES.items():
    for name in names:
        if not os.path.exists(os.path.join(MODEL_ROOT, folder, name)):
            hf_hub_download(repo, name, revision=revision, local_dir=os.path.join(MODEL_ROOT, folder))

os.makedirs(MODEL_ROOT, exist_ok=True)
for name, (url, digest) in CROP_FILES.items():
    path = os.path.join(MODEL_ROOT, name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
    if sha256(path) != digest:
        raise RuntimeError(f"checksum mismatch: {path}")

## 3 · Inputs

In [ ]:
GARMENT_IMAGE = ""  #@param {type:"string"}
PERSON_IMAGE = ""  #@param {type:"string"}
SEED = None  #@param {type:"raw"}
MAX_RES = None  #@param {type:"raw"}
REFERENCE_IMAGE = "/content/reference.jpg"  #@param {type:"string"}
OUTPUT_IMAGE = "/content/tryon.jpg"  #@param {type:"string"}

## 4 · Load

In [ ]:
import ctypes
import glob
import hashlib
import json
import os
import secrets
import site
import time
import urllib.request

for d in sorted({d for p in site.getsitepackages() for d in glob.glob(os.path.join(p, "nvidia", "*", "lib"))}):
    for f in os.listdir(d):
        if ".so" in f and any(k in f for k in ("cudart", "cublas", "cudnn", "cufft", "curand")):
            try:
                ctypes.CDLL(os.path.join(d, f), mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass

import cv2
import mediapipe as mp
import numpy as np
import onnxruntime as ort
import torch
from diffusers import Flux2KleinPipeline, Flux2Transformer2DModel
from IPython.display import display
from mediapipe.tasks import python as mpp
from mediapipe.tasks.python import vision
from PIL import Image, ImageOps

transformer = Flux2Transformer2DModel.from_pretrained(
    os.path.join(MODEL_ROOT, PHOTOROOM[2]), subfolder="transformer_bf16", torch_dtype=torch.bfloat16)
pipe = Flux2KleinPipeline.from_pretrained(
    os.path.join(MODEL_ROOT, BFL[2]), transformer=transformer, torch_dtype=torch.bfloat16).to("cuda")

CUDA = [("CUDAExecutionProvider", {"arena_extend_strategy": "kSameAsRequested",
                                   "cudnn_conv_use_max_workspace": "0",
                                   "do_copy_in_default_stream": "1"}), "CPUExecutionProvider"]
birefnet_options = ort.SessionOptions()
birefnet_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
birefnet_options.intra_op_num_threads = 0
birefnet = ort.InferenceSession(os.path.join(MODEL_ROOT, "BiRefNet_lite.onnx"), birefnet_options, providers=CUDA)
parser = ort.InferenceSession(os.path.join(MODEL_ROOT, "parsing_atr.onnx"), providers=CUDA)
for session in (birefnet, parser):
    if session.get_providers()[0] != "CUDAExecutionProvider":
        raise RuntimeError("an ONNX model did not load on the GPU")

segmenter = vision.ImageSegmenter.create_from_options(vision.ImageSegmenterOptions(
    base_options=mpp.BaseOptions(model_asset_path=os.path.join(MODEL_ROOT, "selfie_multiclass_256x256.tflite")),
    running_mode=vision.RunningMode.IMAGE, output_category_mask=False, output_confidence_masks=True))
poser = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
    base_options=mpp.BaseOptions(model_asset_path=os.path.join(MODEL_ROOT, "pose_landmarker_lite.task")),
    running_mode=vision.RunningMode.IMAGE))

## 5 · Pipeline

In [ ]:
MAXPIX = 1_150_000
AREA = 1_048_576
BALD_PROMPT = ("Make this person completely bald. Remove all hair from the head and any "
               "hair falling over the shoulders, chest or back, and show the scalp. "
               "Keep the clothing, the body, the pose and the background exactly as "
               "they are.")
ER_PROMPT = ("Replace the clothing in image 1 with the clothing in image 2. Keep the person's "
             "face, identity, body and the background exactly as they are.")
BALD_SEED = 46
BIREF_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
BIREF_STD = np.array([0.229, 0.224, 0.225], np.float32)
SCHP_MEAN = np.array([0.406, 0.456, 0.485], np.float32)
SCHP_STD = np.array([0.225, 0.224, 0.229], np.float32)
HAIR, FACE, CLOTHES = 1, 3, 4
ATR_HEAD = (1, 2, 3, 11)
NOSE, L_EAR, R_EAR, L_SH, R_SH = 0, 7, 8, 11, 12
SPECK, GAIN, PAD = 0.01, 1.6, 0.03


def jpeg(bgr):
    return cv2.imdecode(cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])[1], cv2.IMREAD_COLOR)


def normalise(bgr):
    h, w = bgr.shape[:2]
    if h * w <= MAXPIX:
        return bgr
    k = (MAXPIX / (h * w)) ** 0.5
    return cv2.resize(bgr, (int(w * k), int(h * k)), interpolation=cv2.INTER_AREA)


def canvas_reference(bgr):
    h, w = bgr.shape[:2]
    k = min(1.0, (AREA / (h * w)) ** 0.5)
    return max(16, int(h * k) // 16 * 16), max(16, int(w * k) // 16 * 16)


def canvas_tryon(bgr, max_res=None):
    h, w = bgr.shape[:2]
    k = (AREA / (h * w)) ** 0.5
    h, w = max(32, int(h * k) // 32 * 32), max(32, int(w * k) // 32 * 32)
    if max_res and max(h, w) > max_res:
        s = max_res / max(h, w)
        h, w = max(32, int(h * s) // 32 * 32), max(32, int(w * s) // 32 * 32)
    return h, w


def klein(images, prompt, seed, size):
    h, w = size
    out = pipe(prompt=prompt, image=[Image.fromarray(cv2.cvtColor(b, cv2.COLOR_BGR2RGB)) for b in images],
               height=h, width=w, num_inference_steps=4, guidance_scale=0.0,
               generator=torch.Generator("cpu").manual_seed(int(seed))).images[0]
    torch.cuda.synchronize()
    return cv2.cvtColor(np.asarray(out), cv2.COLOR_RGB2BGR)


def mp_image(bgr):
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)))


def largest_cc(mask):
    n, lab, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
    if n <= 1:
        return mask
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return ((lab == k).astype(np.uint8) * 255)


def drop_specks(alpha, share=SPECK):
    b = (alpha > 0.5).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(b, 8)
    if n <= 1:
        return alpha
    tot = max(1, int(b.sum()))
    keep = np.zeros(n, bool)
    keep[1:] = st[1:, cv2.CC_STAT_AREA] >= share * tot
    if not keep.any():
        return alpha
    k = int(max(5, 0.006 * min(alpha.shape))) | 1
    m = cv2.dilate((keep[lab]).astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
    return alpha * m


def refine_band(bgr, prob, band_px, eps=1e-4):
    band_px = int(max(2, band_px))
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * band_px + 1,) * 2)
    hard = (prob > 0.5).astype(np.uint8)
    fg = cv2.erode(hard, k) > 0
    bg = cv2.dilate(hard, k) == 0
    a = cv2.ximgproc.guidedFilter(bgr, prob.astype(np.float32), max(2, band_px), eps)
    a = np.clip((a - 0.5) * GAIN + 0.5, 0.0, 1.0)
    a[fg] = 1.0
    a[bg] = 0.0
    return a


def bbox_of(mask, shape, pad=PAD):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return 0, 0, shape[1], shape[0]
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    m = int(round(pad * max(x1 - x0, y1 - y0)))
    return (max(0, x0 - m), max(0, y0 - m), min(shape[1], x1 + m + 1), min(shape[0], y1 + m + 1))


def matte(bgr):
    h, w = bgr.shape[:2]
    rgb = cv2.cvtColor(cv2.resize(bgr, (1024, 1024), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
    x = (rgb.astype(np.float32) / 255.0 - BIREF_MEAN) / BIREF_STD
    x = np.ascontiguousarray(x.transpose(2, 0, 1)[None])
    y = birefnet.run(None, {birefnet.get_inputs()[0].name: x})[0][0, 0]
    p = 1.0 / (1.0 + np.exp(-y.astype(np.float32)))
    p = cv2.resize(p, (w, h), interpolation=cv2.INTER_CUBIC)
    return np.clip(p, 0.0, 1.0)


def parse_human(bgr):
    h, w = bgr.shape[:2]
    x = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_LINEAR)
    x = ((x.astype(np.float32) / 255.0 - SCHP_MEAN) / SCHP_STD)
    o = parser.run(None, {parser.get_inputs()[0].name: x.transpose(2, 0, 1)[None]})[0][0]
    up = np.stack([cv2.resize(c, (w, h), interpolation=cv2.INTER_LINEAR) for c in o])
    return up.argmax(0).astype(np.uint8)


def neck_line(bgr):
    h, w = bgr.shape[:2]
    r = poser.detect(mp_image(bgr))
    if not r.pose_landmarks:
        return None
    p = r.pose_landmarks[0]
    ear_y = (p[L_EAR].y + p[R_EAR].y) / 2 * h
    sh_y = (p[L_SH].y + p[R_SH].y) / 2 * h
    nose = (p[NOSE].x * w, p[NOSE].y * h)
    return ear_y + (sh_y - ear_y) * 0.72, nose


def head_from_parser(bgr, subject, clothes):
    m = np.isin(parse_human(bgr), ATR_HEAD) & (subject > 0.5)
    m = m & (clothes < 0.5)
    nl = neck_line(bgr)
    if nl is not None:
        cut, nose = nl
        m = m.copy()
        m[int(np.clip(cut, 0, bgr.shape[0] - 1)):] = False
        if m.any():
            n, lab = cv2.connectedComponents(m.astype(np.uint8), 8)
            k = lab[int(np.clip(nose[1], 0, bgr.shape[0] - 1)), int(np.clip(nose[0], 0, bgr.shape[1] - 1))]
            if k > 0:
                m = lab == k
    head = np.clip(cv2.GaussianBlur(m.astype(np.float32), (0, 0), 2.0), 0, 1)
    return head if head.sum() > 40 else None


def head_from_pose(bgr, subject, clothes_prob):
    h, w = bgr.shape[:2]
    r = poser.detect(mp_image(bgr))
    if not r.pose_landmarks:
        return None, False
    p = r.pose_landmarks[0]
    px = lambda i: (p[i].x * w, p[i].y * h)
    (lx, ly), (rx_, ry_) = px(L_EAR), px(R_EAR)
    (nx, ny) = px(NOSE)
    shy = (px(L_SH)[1] + px(R_SH)[1]) / 2.0
    ear_x, ear_y = (lx + rx_) / 2.0, (ly + ry_) / 2.0
    ear_sep = float(np.hypot(lx - rx_, ly - ry_))
    neck = abs(shy - ear_y)
    row = int(np.clip(ear_y, 0, h - 1))
    line = (subject[row] > 0.5)
    xs_on = np.where(line)[0]
    sil_w = 0.0
    if len(xs_on):
        c = int(np.clip((lx + rx_) / 2.0, 0, w - 1))
        b = np.split(xs_on, np.where(np.diff(xs_on) > 1)[0] + 1)
        for seg in b:
            if seg[0] - 4 <= c <= seg[-1] + 4:
                sil_w = float(seg[-1] - seg[0])
                break
    half_w = max(ear_sep * 0.78, neck * 0.62, min(sil_w * 0.58, neck * 0.95), 12.0)
    half_h = half_w * 1.28
    cy = ear_y - half_h * 0.18
    cx = (ear_x + nx) / 2.0
    m = np.zeros((h, w), np.uint8)
    cv2.ellipse(m, (int(cx), int(cy)), (int(half_w), int(half_h)), 0, 0, 360, 1, -1)
    chin = int(ear_y + max(neck, 8.0) * 0.55)
    m[chin:] = 0
    keep = (subject > 0.5)
    below = np.zeros_like(m, dtype=bool)
    below[int(np.clip(ear_y, 0, h - 1)):] = True
    keep = keep & ~((clothes_prob > 0.5) & below)
    a = cv2.GaussianBlur((m.astype(np.float32)) * keep, (0, 0), 2.0)
    return np.clip(a, 0, 1), True


def with_cranium(subject, head, face, clothes_prob):
    fm = (face > 0.4).astype(np.uint8)
    if fm.sum() < 60:
        return head, False
    ys, xs = np.where(fm > 0)
    fy1 = int(np.percentile(ys, 97))
    fx0, fx1 = int(np.percentile(xs, 2)), int(np.percentile(xs, 98))
    fy0 = int(np.percentile(ys, 3))
    fw, fh = max(fx1 - fx0, 8), max(fy1 - fy0, 8)
    m = int(fw * 0.42)
    top = max(0, fy1 - int(1.8 * fh))
    head_h = fy1 - top
    band = np.zeros_like(fm)
    cv2.ellipse(band, ((fx0 + fx1) // 2, int(fy1 - head_h * 0.52)),
                (int((fw / 2 + m) * 0.95), int(head_h * 0.56)), 0, 0, 360, 1, -1)
    skull = (band > 0) & (subject > 0.5)
    if skull.sum() < 40:
        return head, False
    skull = largest_cc((skull.astype(np.uint8) * 255)) > 0
    skull = skull & (clothes_prob <= 0.5)
    if skull.sum() < 40:
        return head, False
    soft = cv2.GaussianBlur(skull.astype(np.float32), (0, 0), 2.0)
    return np.clip(np.maximum(head, soft), 0, 1), True


def head_subtract(bgr):
    h, w = bgr.shape[:2]
    subject = drop_specks(matte(bgr))
    res = segmenter.segment(mp_image(bgr))
    p = np.stack([cv2.resize(m.numpy_view(), (w, h), interpolation=cv2.INTER_LINEAR) for m in res.confidence_masks])
    band = max(3, 0.010 * min(h, w))
    head = refine_band(bgr, p[HAIR] + p[FACE], band)
    face = refine_band(bgr, p[FACE], band)
    parsed = head_from_parser(bgr, subject, p[CLOTHES])
    if parsed is not None:
        head, route = np.clip(np.maximum(head, parsed), 0, 1), "parser"
    else:
        posed, ok = head_from_pose(bgr, subject, p[CLOTHES])
        if ok:
            head, route = np.clip(np.maximum(head, posed), 0, 1), "pose"
        else:
            head, used = with_cranium(subject, head, face, p[CLOTHES])
            route = "face band" if used else "none"
    noface = drop_specks(subject * (1.0 - head))
    x0, y0, x1, y1 = bbox_of((subject > 0.5).astype(np.uint8), bgr.shape[:2])
    a = np.clip(noface[y0:y1, x0:x1], 0, 1)[..., None].astype(np.float32)
    crop = bgr[y0:y1, x0:x1].astype(np.float32) * a + np.float32(255.0) * (1.0 - a)
    return np.clip(crop, 0, 255).astype(np.uint8), route


CACHE_DIR = "/content/cache"
PIPELINE_VERSION = f"er-{BFL[1][:7]}-{PHOTOROOM[1][:7]}"
MIN_PERSON_PIXELS = 524_288


def load_image(spec, label):
    path = spec
    if spec.startswith("http://") or spec.startswith("https://"):
        path = os.path.join("/content", label + (os.path.splitext(spec.split("?")[0])[1] or ".jpg"))
        urllib.request.urlretrieve(spec, path)
    elif not spec or not os.path.exists(spec):
        print(f"choose the {label} photo")
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError(f"no {label} photo uploaded")
        path = os.path.abspath(list(uploaded)[0])
    raw = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if raw is None:
        raise RuntimeError(f"cannot read the {label} photo at {path}")
    if raw.ndim == 3 and raw.shape[2] in (2, 4):
        rgba = np.asarray(ImageOps.exif_transpose(Image.open(path)).convert("RGBA"), np.float32)
        a = rgba[:, :, 3:] / 255.0
        flat = (rgba[:, :, :3] * a + 255.0 * (1.0 - a)).astype(np.uint8)
        return cv2.cvtColor(flat, cv2.COLOR_RGB2BGR)
    return cv2.imread(path, cv2.IMREAD_COLOR)


def garment_key(bgr):
    h = hashlib.sha256()
    h.update(PIPELINE_VERSION.encode())
    h.update(np.ascontiguousarray(bgr).tobytes())
    return h.hexdigest()[:16]


def prepare_garment(garment):
    key = garment_key(garment)
    os.makedirs(CACHE_DIR, exist_ok=True)
    cached_image = os.path.join(CACHE_DIR, key + ".jpg")
    cached_meta = os.path.join(CACHE_DIR, key + ".json")
    if os.path.exists(cached_image) and os.path.exists(cached_meta):
        return cv2.imread(cached_image, cv2.IMREAD_COLOR), json.load(open(cached_meta))["head_route"], {}, True
    reference, route, times = build_reference(garment)
    cv2.imwrite(cached_image, reference, [cv2.IMWRITE_JPEG_QUALITY, 95])
    json.dump({"head_route": route, "pipeline": PIPELINE_VERSION}, open(cached_meta, "w"))
    return reference, route, times, False


def build_reference(garment):
    times = {}
    t = time.perf_counter()
    photo = jpeg(normalise(garment))
    times["normalise"] = time.perf_counter() - t
    t = time.perf_counter()
    bald = klein([photo], BALD_PROMPT, BALD_SEED, canvas_reference(photo))
    bald = jpeg(cv2.resize(bald, (photo.shape[1], photo.shape[0]), interpolation=cv2.INTER_AREA))
    times["bald pass"] = time.perf_counter() - t
    t = time.perf_counter()
    reference, route = head_subtract(bald)
    reference = jpeg(reference)
    times["head crop"] = time.perf_counter() - t
    return reference, route, times


def try_on(person, reference, seed, max_res=None):
    if person.shape[0] * person.shape[1] < MIN_PERSON_PIXELS:
        raise RuntimeError(f"person photo is {person.shape[1]}x{person.shape[0]};"
                           f" below the {MIN_PERSON_PIXELS / 1e6:.1f} MP minimum")
    times = {}
    t = time.perf_counter()
    photo = jpeg(normalise(person))
    times["normalise"] = time.perf_counter() - t
    t = time.perf_counter()
    result = klein([photo, reference], ER_PROMPT, seed, canvas_tryon(photo, max_res))
    times["try-on"] = time.perf_counter() - t
    return result, times

## 6a · Run — prepare the garment (once per garment, cached)

In [ ]:
garment = load_image(GARMENT_IMAGE, "garment")
reference, head_route, garment_times, garment_cached = prepare_garment(garment)
cv2.imwrite(REFERENCE_IMAGE, reference, [cv2.IMWRITE_JPEG_QUALITY, 95])

## 6b · Run — try it on (once per request)

In [ ]:
person = load_image(PERSON_IMAGE, "person")
seed = SEED if SEED is not None else secrets.randbelow(2 ** 31)
result, tryon_times = try_on(person, reference, seed, MAX_RES)
cv2.imwrite(OUTPUT_IMAGE, result, [cv2.IMWRITE_JPEG_QUALITY, 95])

## 7 · Output

In [ ]:
display(Image.fromarray(cv2.cvtColor(result, cv2.COLOR_BGR2RGB)))
print(f"seed           {seed}")
print(f"output         {result.shape[1]}x{result.shape[0]}  {OUTPUT_IMAGE}")
print(f"reference      {reference.shape[1]}x{reference.shape[0]}  {REFERENCE_IMAGE}")
print(f"head route     {head_route}")
print(f"garment        {'from cache' if garment_cached else 'prepared now'}")
for stage, seconds in [*[("garment " + k, v) for k, v in garment_times.items()],
                       *[("person " + k, v) for k, v in tryon_times.items()]]:
    print(f"{stage:22s} {seconds:6.2f} s")
print("rerun section 6b for a new seed; section 6a is cached per garment")